# US 3.3 -- Defect-Aware Enhancement

The baseline CycleGAN (US 3.2) translates AOI images to USM style, but it has
no explicit incentive to **preserve defect regions** during translation.  Small
defects can be smoothed out or distorted by the generator, which defeats the
purpose of translating labeled AOI images.

This notebook adds a **defect preservation loss** that penalises the generator
for altering regions marked as defects in the AOI label masks.

## What you will learn

1. Why baseline CycleGAN can lose defect information
2. The defect preservation loss formulation
3. Comparing baseline vs defect-aware translation
4. How to enable defect-aware training via `lambda_defect`

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic3]"`
- Baseline CycleGAN trained (see `epic3_02_training.ipynb`)

---
## 1. The Problem: Defect Degradation During Translation

CycleGAN optimises for overall image realism and cycle consistency.  But
defect regions are typically small (a few percent of the image) and visually
subtle.  The generator may:

- **Smooth out** small voids to make the image look more realistic
- **Shift** defect boundaries during the style transfer
- **Hallucinate** textures that obscure the original defect pattern

Since we want to transfer AOI labels to the translated USM images, preserving
the exact defect location and shape is critical.

### Solution: Defect Preservation Loss

We add a Dice-based loss that compares the defect mask before and after
translation:

$$L_{defect} = 1 - \text{Dice}(M_{original}, M_{translated})$$

where:
- $M_{original}$ is the known defect mask for the AOI image
- $M_{translated}$ is the defect mask extracted from the translated USM image
  (using a pre-trained defect detector or thresholding)

In [ ]:
import torch
import torch.nn as nn

from udm_epic3.models.cyclegan import CycleGANModel
from udm_epic3.models.losses import (
    adversarial_loss_lsgan,
    cycle_consistency_loss,
    identity_loss,
    defect_preservation_loss,
)

---
## 2. Defect Preservation Loss in Detail

The `defect_preservation_loss` function compares the defect mask regions
between the original AOI image and its translated version.

In [ ]:
# Simulate an AOI image, its translated version, and a defect mask
real_A = torch.randn(2, 3, 256, 256)       # original AOI
fake_B = torch.randn(2, 3, 256, 256)       # translated to USM
defect_mask = torch.zeros(2, 1, 256, 256)  # defect mask
defect_mask[:, :, 100:150, 100:150] = 1.0  # simulate a defect region

# Compute defect preservation loss
loss_defect = defect_preservation_loss(real_A, fake_B, defect_mask)
print(f"Defect preservation loss: {loss_defect.item():.4f}")
print(f"\nThis loss is minimised when defect regions are preserved during translation.")
print(f"A value of 0 means perfect preservation; 1 means total loss of defect info.")

---
## 3. Baseline vs Defect-Aware Model

Let's create two models: one baseline (no defect loss) and one defect-aware.

In [ ]:
# Baseline CycleGAN: no defect preservation
model_baseline = CycleGANModel(
    in_channels=3,
    out_channels=3,
    n_residual_blocks=9,
    n_discriminator_layers=3,
    lambda_cycle=10.0,
    lambda_identity=0.5,
    lambda_defect=0.0,       # no defect loss
)

# Defect-aware CycleGAN: with defect preservation
model_defect = CycleGANModel(
    in_channels=3,
    out_channels=3,
    n_residual_blocks=9,
    n_discriminator_layers=3,
    lambda_cycle=10.0,
    lambda_identity=0.5,
    lambda_defect=5.0,       # defect preservation enabled
)

print(f"Baseline model: lambda_defect = {model_baseline.lambda_defect}")
print(f"Defect-aware model: lambda_defect = {model_defect.lambda_defect}")

---
## 4. Visualising Defect Mask Preservation

After training, we can compare how well each model preserves defect regions
during translation.

In [ ]:
import matplotlib.pyplot as plt

# NOTE: in practice, load trained checkpoints here.
# This is a placeholder showing the visualization pattern.

def visualize_defect_preservation(real_A, fake_B, mask, title=""):
    """Show original AOI, translated USM, and defect mask overlay."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    img_a = (real_A[0] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).detach().numpy()
    img_b = (fake_B[0] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).detach().numpy()
    mask_np = mask[0, 0].detach().numpy()
    
    axes[0].imshow(img_a)
    axes[0].set_title("Original AOI")
    axes[0].axis("off")
    
    axes[1].imshow(img_b)
    axes[1].set_title("Translated USM")
    axes[1].axis("off")
    
    axes[2].imshow(img_b)
    axes[2].imshow(mask_np, alpha=0.4, cmap="Reds")
    axes[2].set_title("USM + Defect Mask Overlay")
    axes[2].axis("off")
    
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()

# Demo with random data (replace with real model outputs after training)
real_A = torch.randn(1, 3, 256, 256)
fake_B = torch.randn(1, 3, 256, 256)
mask = torch.zeros(1, 1, 256, 256)
mask[:, :, 80:130, 90:140] = 1.0

visualize_defect_preservation(real_A, fake_B, mask, title="Defect-Aware Translation")

---
## 5. Total Loss with Defect Preservation

The total generator loss becomes:

$$L = L_{adv} + \lambda_{cyc} \cdot L_{cyc} + \lambda_{idt} \cdot L_{idt} + \lambda_{defect} \cdot L_{defect}$$

Recommended values for `lambda_defect`:

| Value | Effect |
|-------|--------|
| 0.0 | Baseline (no defect preservation) |
| 1.0 | Mild defect preservation |
| 5.0 | Strong defect preservation (recommended starting point) |
| 10.0 | Very strong -- may reduce translation quality |

Finding the right balance is important: too high and the generator cannot
change defect regions at all (defeating the style transfer), too low and
defects are lost.

---
## 6. CLI: Training with Defect Preservation

Enable defect-aware training by setting `lambda_defect > 0` in the config:

```bash
# Train with defect preservation
udm-epic3 train --config configs/epic3_cyclegan.yaml
```

Config changes for defect-aware training:

```yaml
training:
  lambda_cycle: 10.0
  lambda_identity: 0.5
  lambda_defect: 5.0       # enable defect preservation

data:
  mask_dir: data/epic3/masks  # AOI defect masks (required when lambda_defect > 0)
```

The training script automatically detects `lambda_defect > 0` and loads
defect masks from the specified directory.

---
## Next steps

| Notebook | Topic |
|----------|------|
| `epic3_04_evaluation.ipynb` | Quantitative comparison of baseline vs defect-aware |
| `epic3_05_downstream.ipynb` | Train segmentation on translated data |